### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lung_cancer",
    dataset_year="2001",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other", # supplementary materials of the paper
    original_dataset_source_download_link="https://www.pnas.org/doi/10.1073/pnas.191502998#supplementary-materials",
    download_description=r"""
Cannot download automatically!

Click and download yourself: "https://www.pnas.org/doi/suppl/10.1073/pnas.191502998/suppl_file/dataseta_12600gene.xls"
mkdir -p local-data-warehouse/lung_cancer/ && mv dataseta_12600gene.xls local-data-warehouse/lung_cancer/
""",
    # References
    academic_reference_bibtex=r"""@article{bhattacharjee2001classification,
  title={Classification of human lung carcinomas by mRNA expression profiling reveals distinct adenocarcinoma subclasses},
  author={Bhattacharjee, Arindam and Richards, William G and Staunton, Jane and Li, Cheng and Monti, Stefano and Vasa, Priya and Ladd, Christine and Beheshti, Javad and Bueno, Raphael and Gillette, Michael and others},
  journal={Proceedings of the National Academy of Sciences},
  volume={98},
  number={24},
  pages={13790--13795},
  year={2001},
  publisher={The National Academy of Sciences}
}
""",
    academic_reference_bibtex_key="bhattacharjee2001classification",
    license="None", # not given as part of data download on the website, likely Copyright for the journal
    data_tags=["IID"],
    curation_comments="""
- The original data comes with lung adenocarcinoma and other adenocarcinomas merged into one class already. We keep the same and were not able to find a way to reverse this from the public data.
- We also drop the class "small-cell lung carcinoma" as it only has 6 samples, which is not a meaningful sample size for benchmarking, or the predictive task.  We could add another version with all classes in the future.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="CancerType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="CancerType",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_excel(f"{dataset_mold.path}/dataseta_12600gene.xls")

# Translate into tabular format
df = (
    df
    .set_index("probe set")        # use probe set as row index → becomes columns after .T
    .drop(columns=["gene"])        # drop unwanted column
    .T                             # transpose
    .reset_index(names="CancerType")
)
# Extract classes from index
prefix_to_class = {
    "AD": "lung adenocarcinoma",
    "SQ": "squamous cell carcinoma",
    "COID": "pulmonary carcinoid",
    "SMCL": "small-cell lung carcinoma",
    "NL": "normal lung"
}
# extract prefix and map to class
df["CancerType"] = df["CancerType"].str.split("-").str[0].map(prefix_to_class)

df = df[df["CancerType"] != "small-cell lung carcinoma"]
df["CancerType"] = df["CancerType"].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 197
Columns: 12601
Use sampling: False (sample size: 197)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['AFFX-BioB-3_st', 'AFFX-DapX-5_at', 'AFFX-hum_alu_at', 'AFFX-MurIL2_at', 'AFFX-ThrX-5_at', 'AFFX-HUMISGF3A/M97935_3_at', '39733_at', '39721_at', '39722_at', '39750_at']
Rows remaining as candidates after top-10 filter: 0 (of 197)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

probe set,CancerType,AFFX-MurIL2_at,AFFX-MurIL10_at,AFFX-MurIL4_at,AFFX-MurFAS_at,AFFX-BioB-5_at,AFFX-BioB-M_at,AFFX-BioB-3_at,AFFX-BioC-5_at,AFFX-BioC-3_at,AFFX-BioDn-5_at,AFFX-BioDn-3_at,AFFX-CreX-5_at,AFFX-CreX-3_at,AFFX-BioB-5_st,AFFX-BioB-M_st,AFFX-BioB-3_st,AFFX-BioC-5_st,AFFX-BioC-3_st,AFFX-BioDn-5_st,AFFX-BioDn-3_st,AFFX-CreX-5_st,AFFX-CreX-3_st,AFFX-hum_alu_at,AFFX-DapX-5_at,AFFX-DapX-M_at,AFFX-DapX-3_at,AFFX-LysX-5_at,AFFX-LysX-M_at,AFFX-LysX-3_at,AFFX-PheX-5_at,AFFX-PheX-M_at,AFFX-PheX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,AFFX-ThrX-3_at,AFFX-TrpnX-5_at,AFFX-TrpnX-M_at,AFFX-TrpnX-3_at,AFFX-HUMISGF3A/M97935_5_at,AFFX-HUMISGF3A/M97935_MA_at,AFFX-HUMISGF3A/M97935_MB_at,AFFX-HUMISGF3A/M97935_3_at,AFFX-HUMRGE/M10098_5_at,AFFX-HUMRGE/M10098_M_at,AFFX-HUMRGE/M10098_3_at,AFFX-HUMGAPDH/M33197_5_at,AFFX-HUMGAPDH/M33197_M_at,AFFX-HUMGAPDH/M33197_3_at,AFFX-HSAC07/X00351_5_at,AFFX-HSAC07/X00351_M_at,AFFX-HSAC07/X00351_3_at,AFFX-HUMTFRR/M11507_5_at,AFFX-HUMTFRR/M11507_M_at,AFFX-HUMTFRR/M11507_3_at,AFFX-M27830_5_at,AFFX-M27830_M_at,AFFX-M27830_3_at,AFFX-HSAC07/X00351_3_st,AFFX-HUMGAPDH/M33197_5_st,AFFX-HUMGAPDH/M33197_M_st,AFFX-HUMGAPDH/M33197_3_st,AFFX-HSAC07/X00351_5_st,AFFX-HSAC07/X00351_M_st,AFFX-YEL002c/WBP1_at,AFFX-YEL018w/_at,AFFX-YEL024w/RIP1_at,AFFX-YEL021w/URA3_at,31307_at,31308_at,31309_r_at,31310_at,31311_at,31312_at,31313_at,31314_at,31315_at,31316_at,31317_r_at,31318_at,31319_at,31320_at,31321_at,31322_at,31323_r_at,31324_at,31325_at,31326_at,31327_at,31328_at,31329_at,31330_at,31331_at,31332_at,31333_at,31334_at,31335_at,31336_at,31337_at,31338_at,31339_at,31340_at,31341_at,31342_at,31343_at,31344_at,31345_at,31346_at,31347_at,31348_at,31349_at,31350_at,31351_at,31352_at,31353_f_at,31354_r_at,31355_at,31356_at,31357_at,31358_at,31359_at,31360_at,31361_at,31362_at,31363_at,31364_i_at,31365_f_at,31366_at,31367_at,31368_at,31369_at,31370_at,31371_at,31372_at,31373_at,31374_at,31375_at,31376_at,31377_r_at,31378_at,31379_at,31380_at,31381_at,31382_f_at,31383_at,31384_at,31385_at,31386_at,31387_at,31388_at,31389_at,31390_at,31391_at,31392_r_at,31393_r_at,31394_at,31395_i_at,31396_r_at,31397_at,31398_at,31399_at,31400_at,31401_r_at,31402_at,31403_at,31404_at,31405_at,31406_at,31407_at,31408_at,31409_at,31410_at,31411_at,31412_at,31413_at,31414_at,31415_at,31416_at,31417_at,31418_at,31419_r_at,31420_at,31421_at,31422_at,31423_at,31424_at,31425_g_at,31426_at,31427_at,31428_at,31429_at,31430_at,31431_at,31432_g_at,31433_at,31434_at,31435_at,31436_s_at,31437_r_at,31438_s_at,31439_f_at,31440_at,31441_at,31442_at,31443_at,31444_s_at,31445_at,31446_s_at,31447_at,31448_s_at,31449_at,31450_s_at,31451_at,31452_at,31453_s_at,31454_f_at,31455_r_at,31456_at,31457_at,31458_at,31459_i_at,31460_f_at,31461_at,31462_f_at,31463_s_at,31464_at,31465_g_at,31466_at,31467_at,31468_f_at,31469_s_at,31470_at,31471_at,31472_s_at,31473_s_at,31474_r_at,31475_at,31476_g_at,31477_at,31478_at,31479_f_at,31480_f_at,31481_s_at,31482_at,31483_g_at,31484_at,31485_at,31486_s_at,31487_at,31488_s_at,31489_at,31490_at,31491_s_at,31492_at,31493_s_at,31494_at,31495_at,31496_g_at,31497_at,31498_f_at,31499_s_at,31500_at,31501_at,31502_at,31503_at,31504_at,31505_at,31506_s_at,31507_at,31508_at,31509_at,31510_s_at,31511_at,31512_at,31513_at,31514_at,31515_at,31516_f_at,31517_f_at,31518_i_at,31519_f_at,31520_at,31521_f_at,31522_f_at,31523_f_at,31524_f_at,31525_s_at,31526_f_at,31527_at,31528_f_at,31529_at,31530_at,31531_g_at,31532_at,31533_s_at,31534_at,31535_i_at,31536_at,31537_at,31538_at,31539_r_at,31540_at,31541_at,31542_at,31543_at,31544_at,31545_at,31546_at,31547_at,31548_at,31549_at,31550_at,31551_at,31552_at,31553_at,31554_at,31555_at,31556_at,31557_at,31558_at,31559_at,31560_at,31561_at,31562_at,31563_at,31564_at,31565_at,31566_at,31567_at,31568_at,31569_at,31570_at,31571_at,31572_at,31573_at,31574_i_at,31575_f_at,31576_at,31577_at,31578_at,31579_at,31580_at,31581_at,31582_at,31583_at,31584_at,31585_at,31586_f_at,31587_at,31588_at,31589_at,31590_g_at,31591_s_at,315

In [5]:
# Feature Summary
summary.head(100) # limit number of cols we show...

,index,dtype,n_missing,pct_missing,n_unique,examples
0,CancerType,category,0.0,0.0,4.0,"lung adenocarcinoma, squamous cell carcinoma, pulmonary carcinoid, normal lung"
1,AFFX-MurIL2_at,float64,0.0,0.0,197.0,"-29.4, 3.01, -44.73, -34.15, -14.29, -0.81, 27.95, -25.49, -21.165, -15.82"
2,AFFX-MurIL10_at,float64,0.0,0.0,196.0,"11.93, -27.79, -15.36, 14.26, -11.46, -1.17, -12.5, 28.16, -11.535, -7.9"
3,AFFX-MurIL4_at,float64,0.0,0.0,192.0,"14.48, 0.01, 11.06, -13.45, -24.56, -0.23, -3.06, -28.6, -16.415, -12.22"
4,AFFX-MurFAS_at,float64,0.0,0.0,194.0,"7.83, 9.15, 15.68, 13.22, 25.73, 16.26, 38.98, -9.06, 3.99, -7.2"
5,AFFX-BioB-5_at,float64,0.0,0.0,194.0,"-5.54, -25.82, -26.24, -25.16, 5.295, -34.15, 20.6, -24.06, -20.85, -28.17"
6,AFFX-BioB-M_at,float64,0.0,0.0,196.0,"-36.98, -55.15, -39.84, 0.47, -24.91, -12.41, 13.24, -29.78, -28.16, -53.94"
7,AFFX-BioB-3_at,float64,0.0,0.0,194.0,"1.79, -27.9, -14.69, -23.52, 10.06, -22.39, 19.37, -21.2, -14.29, 16.89"
8,AFFX-BioC-5_at,float64,0.0,0.0,196.0,"21.05, -5.27, -13.73, 27.89, 29.69, 44.78, 45.11, 11.26, -3.525, 3.6"
9,AFFX-BioC-3_at,float64,0.0,0.0,196.0,"11.68, -55.96, -32.5, 1.265, -15.23, -40.23, 9.57, -36.2, -28.035, -53.94"


In [6]:
# Numeric Feature Statistics
numeric_stats.head(100) # limit number of cols we show...

,count,mean,std,min,max
AFFX-MurIL2_at,197.0,-8.542568,20.308881,-59.530,92.680
AFFX-MurIL10_at,197.0,4.500651,21.877427,-33.200,106.160
AFFX-MurIL4_at,197.0,0.162090,18.938182,-31.180,70.500
AFFX-MurFAS_at,197.0,19.507065,20.319746,-18.630,91.790
AFFX-BioB-5_at,197.0,-15.592559,19.091591,-67.000,46.340
AFFX-BioB-M_at,197.0,-21.783926,26.261016,-263.470,53.660
AFFX-BioB-3_at,197.0,-4.627035,25.392808,-44.000,210.100
AFFX-BioC-5_at,197.0,22.370042,26.250110,-34.810,114.500
AFFX-BioC-3_at,197.0,-23.943130,16.026964,-57.040,22.295
AFFX-BioDn-5_at,197.0,-53.042064,17.085969,-98.670,16.420


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                                       
CancerType 1         lung adenocarcinoma    139  70.56
           2     squamous cell carcinoma     21  10.66
           3         pulmonary carcinoid     20  10.15
           4                 normal lung     17   8.63

In [8]:
# Target Distribution
target_df

,count,pct
CancerType,,
lung adenocarcinoma,139,70.56
squamous cell carcinoma,21,10.66
pulmonary carcinoid,20,10.15
normal lung,17,8.63


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to lung_cancer/019d6e03-622d-7ca8-b1bf-5ef42a921012
019d6e03-622d-7ca8-b1bf-5ef42a921012
dcbbd1c7784ac8b7eaf9e2b4349ba9d872d2832b46144a77a0768e9d2eea45f9
